<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.4-open-source-fine-tuning/notebooks/exercises-7.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.4 — Open-Source Fine-Tuning
Netsetos GenAI Course v2.0 | Module 7

LoRA/QLoRA math, Unsloth+TRL, DPO/GRPO alignment, deployment pipeline, India infrastructure


In [ ]:
# pip install unsloth trl peft bitsandbytes transformers -q
print('Ready for Open-Source Fine-Tuning')


## Ex 1: LoRA Parameter Calculator


In [ ]:
def lora_params(d, k, r):
    return (d * r) + (r * k)  # B + A matrices

d = k = 4096  # Llama 7B hidden size
total_per_layer = d * k

print('LoRA trainable params for 4096×4096 weight matrix:')
for r in [4, 8, 16, 32, 64, 128]:
    params = lora_params(d, k, r)
    pct = params / total_per_layer * 100
    print(f'  r={r:3d}: {params:>8,} params ({pct:.2f}% of {total_per_layer:,})')


## Ex 2: QLoRA VRAM Estimator


In [ ]:
def vram_estimate(model_params_B, method='full'):
    if method == 'full':
        return model_params_B * 2 * 8  # FP16 weights + optimizer (8× for Adam)
    elif method == 'lora':
        return model_params_B * 2 + 0.5  # FP16 base + small adapter
    elif method == 'qlora':
        return model_params_B * 0.5 + 0.5  # 4-bit base + adapter
    elif method == 'qlora_unsloth':
        return (model_params_B * 0.5 + 0.5) * 0.6  # +40% Unsloth savings

for size in [7, 13, 70]:
    print(f'\n{size}B model VRAM:')
    for m in ['full', 'lora', 'qlora', 'qlora_unsloth']:
        gb = vram_estimate(size, m)
        fits = '✅ T4(16GB)' if gb <= 16 else ('✅ A100(40GB)' if gb <= 40 else '❌ Multi-GPU')
        print(f'  {m:18s}: {gb:6.1f} GB  {fits}')


## Ex 3: Production LoRA Config


In [ ]:
print('2026 Production LoRA Config:')
print()
config = {
    'r': 16,
    'lora_alpha': 32,
    'target_modules': 'all-linear',
    'use_dora': True,
    'use_rslora': True,
    'lora_dropout': 0.0,
    'bias': 'none',
    'task_type': 'CAUSAL_LM',
}
for k, v in config.items():
    print(f'  {k:20s}: {v}')
print()
print('Why each choice:')
print('  r=16: Production sweet spot (Unsloth recommended)')
print('  alpha=32: 2×rank (Sebastian Raschka optimal)')
print('  all-linear: QLoRA paper validated = full FT quality')
print('  DoRA: +1-4% quality, zero inference overhead')
print('  rsLoRA: Stable gradients (essential if r≥32)')


## Ex 4: Unsloth SFT Pipeline


In [ ]:
print('Unsloth + TRL SFT Pipeline:')
print()
print('# 1. Load model with Unsloth')
print('from unsloth import FastLanguageModel')
print('model, tokenizer = FastLanguageModel.from_pretrained(')
print('    model_name="unsloth/gemma-2-2b", max_seq_length=2048,')
print('    load_in_4bit=True)')
print()
print('# 2. Add LoRA adapters')
print('model = FastLanguageModel.get_peft_model(model,')
print('    r=16, lora_alpha=32, target_modules="all-linear",')
print('    use_gradient_checkpointing="unsloth")')
print()
print('# 3. Train with SFTTrainer')
print('from trl import SFTTrainer, SFTConfig')
print('trainer = SFTTrainer(model=model, tokenizer=tokenizer,')
print('    train_dataset=dataset,')
print('    args=SFTConfig(output_dir="./out", num_train_epochs=3,')
print('        per_device_train_batch_size=2,')
print('        gradient_accumulation_steps=8,')
print('        learning_rate=2e-4, report_to="wandb"))')
print('trainer.train()')
print()
print('# 4. Export to GGUF')
print('model.save_pretrained_gguf("model_gguf", tokenizer,')
print('    quantization_method="q4_k_m")')


## Ex 5: DPO Training


In [ ]:
print('DPO Training with TRL:')
print()
print('from trl import DPOTrainer, DPOConfig')
print()
print('# Data format: prompt + chosen + rejected')
print('dpo_data = {')
print('  "prompt": "What color is the sky?",')
print('  "chosen": "The sky is blue due to Rayleigh scattering.",')
print('  "rejected": "The sky is green."')
print('}')
print()
print('# Key hyperparameters')
for k,v in [('beta','0.1 (KL penalty, most important)'),
    ('learning_rate','5e-7 (10× lower than SFT)'),
    ('epochs','1 (rarely need more)'),
    ('optimizer','paged_adamw_8bit')]:
    print(f'  {k:18s}: {v}')
print()
print('# With LoRA: TRL disables adapter for reference logits')
print('# → No separate reference model needed!')


## Ex 6: GRPO Reasoning Training


In [ ]:
print('GRPO Training with TRL:')
print()
print('import re')
print('from trl import GRPOTrainer, GRPOConfig')
print()
print('def format_reward(completions, **kwargs):')
print('    pattern = r"<think>.*?</think>\\s*<answer>.*?</answer>"')
print('    return [1.0 if re.search(pattern, c) else 0.0 for c in completions]')
print()
print('def accuracy_reward(completions, answer, **kwargs):')
print('    rewards = []')
print('    for c in completions:')
print('        match = re.search(r"<answer>(.*?)</answer>", c)')
print('        rewards.append(1.0 if match and match.group(1).strip() == answer else 0.0)')
print('    return rewards')
print()
print('# Key: no reward model needed, just verifiable functions!')
print('# loss_type="dapo": token-level normalization, removes length bias')
print('# num_generations=4: completions per prompt')
print('# use_vllm=True: fast generation during RL')


## Ex 7: Deployment Pipeline


In [ ]:
print('Full deployment pipeline:')
print()
steps = [
    ('1. Merge LoRA', 'model.merge_and_unload() → standard model, zero overhead'),
    ('2a. GGUF (local)', 'save_pretrained_gguf("out", tok, quantization_method="q4_k_m")'),
    ('2b. AWQ (GPU)', 'llm-compressor for INT4 → 741 tok/s with Marlin kernels'),
    ('3a. Ollama', 'Modelfile + ollama create my-model -f Modelfile'),
    ('3b. vLLM', 'vllm serve model --enable-lora --lora-modules name=/path'),
    ('3c. HF Endpoints', 'push_to_hub() + deploy, scale-to-zero, T4 $0.50/hr'),
]
for step, desc in steps:
    print(f'  {step:20s}: {desc}')
print()
print('Multi-LoRA serving (vLLM):')
print('  1 base model (14GB) + 3 adapters (50MB each)')
print('  vs 3 full models (42GB total) → 3× memory savings')
print('  Route per request via model parameter')


## Ex 8: India Cost Calculator


In [ ]:
print('India fine-tuning cost tiers:')
print()
tiers = [
    ('$0 (Free)', 'Colab T4 + Unsloth + Sarvam-1/Gemma-2B', '4-8 hours'),
    ('₹432 (~$5)', 'Jarvislabs A100 40GB × 4hrs', 'QLoRA 7B with iterations'),
    ('₹900 (~$11)', 'Jarvislabs A100 80GB × 8hrs', 'QLoRA 13B-24B models'),
    ('₹2,250 (~$27)', 'IndiaAI H100 × 10hrs @ ₹225/hr', 'QLoRA 70B model'),
    ('₹670 (~$8)', 'IndiaAI subsidized × 10hrs @ ₹67/hr', 'Same but with subsidy!'),
]
for cost, setup, capability in tiers:
    print(f'  {cost:18s}: {setup}')
    print(f'  {"":18s}  → {capability}')
    print()
print('Best base models for Hindi fine-tuning:')
for model, why in [
    ('Sarvam-1 (2B)', 'Best Indic tokenizer (1.4-2.1 fertility)'),
    ('Gemma 3 (4B)', 'Good multilingual tokenizer (~2.2 fertility)'),
    ('Sarvam-M (24B)', 'Best Indic benchmark scores'),
]: print(f'  {model:20s}: {why}')


---
## Recap
LoRA trains 0.5-1.3% of parameters: W'=W₀+(α/r)·BA. QLoRA adds NF4+double quant+paged optimizers: 120GB→6-10GB. Production config: r=16, α=32, all-linear, DoRA, rsLoRA. Unsloth+TRL is the dominant stack (2× speed, 70% VRAM reduction). Axolotl for multi-GPU, LLaMA-Factory for zero-code. Alignment: DPO for preferences (beta=0.1, lr=5e-7), GRPO for reasoning (verifiable rewards, no reward model). Deploy: merge→GGUF Q4_K_M→Ollama (local) or AWQ→vLLM (GPU). vLLM multi-LoRA serves multiple adapters from one base. India: Sarvam models (Apache 2.0), IndiaAI ₹67/GPU-hr, Jarvislabs ₹108/hr, IndicAlign 74.7M pairs.
